In [3]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd

In [4]:
def get_spread_page():
    url = "https://rotogrinders.com/lineups/nfl"
    html = requests.get(url).text
    page = soup(html)
    return page
page = get_spread_page()

team_list = page.find_all("span", {"class": "shrt"})
teams = [i.text for i in team_list]
odds_list = page.find_all("a", {"href": "/nfl/odds"})
odds = [i.text for i in odds_list]
teams.sort()
print(teams)

['ARI', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GBP', 'HOU', 'IND', 'JAC', 'KCC', 'LAC', 'LAR', 'LVR', 'MIA', 'MIN', 'NEP', 'NYG', 'PHI', 'PIT', 'SEA', 'TBB', 'TEN', 'WAS']


In [6]:
df_fd_temp = pd.read_csv("data/nfl/FanDuel-NFL-2021 ET-10 ET-17 ET-65217-players-list.csv")
df_fd = df_fd_temp.convert_dtypes()
df_fd.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
0,65217-55050,RB,Christian,Christian McCaffrey,McCaffrey,16.800001,3,10000,MIN@CAR,CAR,MIN,IR,Hamstring,<NA>,<NA>,<NA>,RB/FLEX
1,65217-57439,QB,Patrick,Patrick Mahomes,Mahomes,27.379999,5,9000,KC@WAS,KC,WAS,<NA>,<NA>,<NA>,<NA>,<NA>,QB
2,65217-54140,RB,Dalvin,Dalvin Cook,Cook,12.866666,3,8800,MIN@CAR,MIN,CAR,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX
3,65217-53681,WR,Tyreek,Tyreek Hill,Hill,19.32,5,8700,KC@WAS,KC,WAS,Q,Quadriceps,<NA>,<NA>,<NA>,WR/FLEX
4,65217-33076,TE,Travis,Travis Kelce,Kelce,15.18,5,8500,KC@WAS,KC,WAS,<NA>,<NA>,<NA>,<NA>,<NA>,TE/FLEX


In [7]:
df_teams = list(df_fd.Team.unique())
df_teams.sort()
print(df_teams)

['ARI', 'BAL', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'KC', 'LAC', 'LAR', 'LV', 'MIN', 'NE', 'NYG', 'WAS']


In [ ]:
replace_dict = {"GBP":"GB", "KCC":"KC", "LVR":"LV", "NEP":"NE", "NOS":"NO", "SFO":"SF", "TBB":"TB"}
clean_teams = [replace_dict.get(item,item) for item in teams]
print(clean_teams)

['ARI', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAC', 'KC', 'LAC', 'LAR', 'LV', 'MIA', 'MIN', 'NE', 'NYG', 'PHI', 'PIT', 'SEA', 'TB', 'TEN', 'WAS']


In [9]:
df_odds = dict(zip(clean_teams, odds))
#print(df_odds)
print(dict(sorted(df_odds.items(), key=lambda item: item[1])))

{'GB': '16.75', 'SEA': '18.75', 'LV': '19.25', 'NE': '20.00', 'JAC': '20.25', 'DET': '21.25', 'CIN': '21.50', 'MIA': '22.50', 'BAL': '22.75', 'CAR': '23.00', 'PIT': '23.50', 'DAL': '23.75', 'DEN': '23.75', 'KC': '23.75', 'TB': '23.75', 'WAS': '23.75', 'NYG': '24.00', 'BUF': '24.50', 'CHI': '25.00', 'LAR': '25.25', 'MIN': '26.00', 'HOU': '26.75', 'LAC': '26.75', 'PHI': '27.00', 'IND': '28.25', 'TEN': '29.25', 'ARI': '29.75', 'CLE': '30.25'}


In [10]:
df_fd_def = df_fd.query("Position == 'D'")
df_fd = df_fd.query("Position != 'D'")
df_fd_def.head(3)

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
198,65217-12538,D,Los Angeles,Los Angeles Rams,Rams,5.0,5,5000,LAR@NYG,LAR,NYG,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
234,65217-12535,D,Indianapolis,Indianapolis Colts,Colts,6.6,5,4700,HOU@IND,IND,HOU,<NA>,<NA>,<NA>,<NA>,<NA>,DEF
252,65217-12537,D,Las Vegas,Las Vegas Raiders,Raiders,4.0,5,4600,LV@DEN,LV,DEN,<NA>,<NA>,<NA>,<NA>,<NA>,DEF


In [11]:
df_fd["expts"] = df_fd["Team"].map(df_odds)
df_fd.expts = df_fd.expts.astype('float')
pts_bins = [1.2, 1.1, 1.0, .9,.8]
df_fd["rkpts"] = pd.cut(df_fd.expts, 5, labels=pts_bins)
df_fd.rkpts = df_fd.rkpts.astype('float')
df_fd["new_expts"] = df_fd.FPPG * df_fd.rkpts
df_fd.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position,expts,rkpts,new_expts
0,65217-55050,RB,Christian,Christian McCaffrey,McCaffrey,16.800001,3,10000,MIN@CAR,CAR,MIN,IR,Hamstring,<NA>,<NA>,<NA>,RB/FLEX,23.00,1.0,16.800001
1,65217-57439,QB,Patrick,Patrick Mahomes,Mahomes,27.379999,5,9000,KC@WAS,KC,WAS,<NA>,<NA>,<NA>,<NA>,<NA>,QB,23.75,1.0,27.379999
2,65217-54140,RB,Dalvin,Dalvin Cook,Cook,12.866666,3,8800,MIN@CAR,MIN,CAR,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX,26.00,0.9,11.58
3,65217-53681,WR,Tyreek,Tyreek Hill,Hill,19.32,5,8700,KC@WAS,KC,WAS,Q,Quadriceps,<NA>,<NA>,<NA>,WR/FLEX,23.75,1.0,19.32
4,65217-33076,TE,Travis,Travis Kelce,Kelce,15.18,5,8500,KC@WAS,KC,WAS,<NA>,<NA>,<NA>,<NA>,<NA>,TE/FLEX,23.75,1.0,15.18


In [12]:
df_fd.FPPG = df_fd.new_expts
df_fd = df_fd.drop(['expts', 'rkpts', 'new_expts'], axis=1)
df_fd = pd.concat([df_fd, df_fd_def])
#df_fd.query("Position == 'D'")
df_fd.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Unnamed: 14,Unnamed: 15,Roster Position
0,65217-55050,RB,Christian,Christian McCaffrey,McCaffrey,16.800001,3,10000,MIN@CAR,CAR,MIN,IR,Hamstring,<NA>,<NA>,<NA>,RB/FLEX
1,65217-57439,QB,Patrick,Patrick Mahomes,Mahomes,27.379999,5,9000,KC@WAS,KC,WAS,<NA>,<NA>,<NA>,<NA>,<NA>,QB
2,65217-54140,RB,Dalvin,Dalvin Cook,Cook,11.58,3,8800,MIN@CAR,MIN,CAR,<NA>,<NA>,<NA>,<NA>,<NA>,RB/FLEX
3,65217-53681,WR,Tyreek,Tyreek Hill,Hill,19.32,5,8700,KC@WAS,KC,WAS,Q,Quadriceps,<NA>,<NA>,<NA>,WR/FLEX
4,65217-33076,TE,Travis,Travis Kelce,Kelce,15.18,5,8500,KC@WAS,KC,WAS,<NA>,<NA>,<NA>,<NA>,<NA>,TE/FLEX


In [13]:
df_fd.to_csv('data/nfl/FDprojections.csv')

In [18]:
from pydfs_lineup_optimizer import Site, Sport, get_optimizer, PlayerFilter, AfterEachExposureStrategy, TeamStack

optimizer = get_optimizer(Site.FANDUEL, Sport.FOOTBALL)
optimizer.load_players_from_csv("data/nfl/FDprojections.csv")
optimizer.add_stack(TeamStack(3, for_positions=['QB', 'WR', 'TE']))
#optimizer.add_stack(TeamStack(2, for_positions=['RB', 'D']))
for lineup in optimizer.optimize(40, max_exposure=0.4):
    print(lineup)
optimizer.export('data/nfl/fd_result.csv')

 1. QB      Lamar Jackson                 QB    BAL            LAC@BAL  26.772         8200.0$   
 2. RB      Aaron Jones                   RB    GB             GB@CHI   19.344         8000.0$   
 3. RB      Josh Jacobs                   RB    LV             LV@DEN   15.76          6800.0$   
 4. WR      Chris Moore                   WR    HOU            HOU@IND  17.46          5300.0$   
 5. WR      Davante Adams                 WR    GB             GB@CHI   21.816         8500.0$   
 6. WR      Hunter Renfrow                WR    LV             LV@DEN   13.56          5700.0$   
 7. TE      Mark Andrews                  TE    BAL            LAC@BAL  14.1           6300.0$   
 8. FLEX    Marquise Brown                WR    BAL            LAC@BAL  17.92          7000.0$   
 9. DEF     Dallas Cowboys                D     DAL            DAL@NE   9.2            4100.0$   

Fantasy Points 155.93
Salary 59900.00

 1. QB      Lamar Jackson                 QB    BAL            LAC@BAL  26.772